# 1. Using the Hugging Face datasets Library to Create Custom Dataset

In [1]:
!pip install pandas librosa numpy datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.7/472.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 12.3 MB/s eta 0:00:00


In [2]:
import os
import pandas as pd
import librosa
import numpy as np
from datasets import Dataset, DatasetDict, Features, Value, Sequence
import string
import re

In [3]:
# You do not need to install google-colab if you are using Google Colab itself.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [29]:
# Set your main directory here
MAIN_DIR = '/content/drive/MyDrive/shayan/turkey_sample'


In [30]:
def get_person_folders(main_dir):
    """Retrieve all person folders ending with '-bot'."""
    return [
        os.path.join(main_dir, folder)
        for folder in os.listdir(main_dir)
        if os.path.isdir(os.path.join(main_dir, folder)) and folder.endswith('-bot')
    ]

In [31]:
def normalize_text(text):
    """Normalize text by lowercasing and removing punctuation."""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = ' '.join(text.split())  # Remove extra whitespace
    return text

In [32]:
def load_csv_files(name_csv_dir):
    """Load CSV files into a dictionary with PersonID as keys."""
    csv_dict = {}
    for file in os.listdir(name_csv_dir):
        if file.endswith('.csv'):
            person_id = os.path.splitext(file)[0]
            # print(person_id)
            csv_path = os.path.join(name_csv_dir, file)
            try:
                df = pd.read_csv(csv_path)
                csv_dict[person_id] = df
            except Exception as e:
                print(f"Error reading {csv_path}: {e}")
    return csv_dict

In [33]:
def create_tts_dataset(main_dir, sampling_rate=22050):
        dataset = []
        person_folders = get_person_folders(main_dir)
        name_csv_dir = os.path.join(main_dir, 'Name_csv')
        csv_dict = load_csv_files(name_csv_dir)

        for person_folder in person_folders:
            # Extract PersonID from folder name (assuming format 'PersonID-bot')
            folder_name = os.path.basename(person_folder)
            print(folder_name, " : folder_name")
            person_id = folder_name.replace('-bot', '')

            if person_id not in csv_dict:
                print(f"CSV for PersonID {person_id} not found. Skipping this person.")
                continue

            df = csv_dict[person_id]

            # Iterate over all wav files in the person folder
            for file in os.listdir(person_folder):
                if file.endswith('.wav'):
                    # Expected filename format: "PersonID_SentenceID_Number_BookID.wav"
                    try:
                        parts = os.path.splitext(file)[0].split('_')
                        if len(parts) != 4:
                            print(f"Filename {file} does not match the expected format. Skipping.")
                            continue
                        file_person_id, sentence_id, number, book_id = parts
                        match = re.search(r'ID(\d+)', sentence_id)
                        if match:
                            sentence_id = int(match.group(1))
                        if 'f' in file_person_id:
                          gender = 'female'
                        else:
                          gender = 'male'

                    except Exception as e:
                        print(f"Error parsing filename {file}: {e}")
                        continue

                    # Verify PersonID matches
                    if file_person_id != person_id:
                        print(f"PersonID mismatch in file {file}. Skipping.")
                        continue

                    # Find the corresponding text in the CSV
                    # Assuming that SentenceID, Number, and BookID uniquely identify a row
                    matching_rows = df.iloc[sentence_id]


                    if matching_rows.empty:
                        print(f"No matching text found for file {file}. Skipping.")
                        continue

                    # text = matching_rows.iloc[0]['Sentence']
                    text = matching_rows['Sentence']
                    normalized = normalize_text(text)

                    # Define the path to the audio file
                    audio_path = os.path.join(person_folder, file)

                    # Load audio file
                    try:
                        audio_array, sr = librosa.load(audio_path, sr=sampling_rate)
                        # Optionally, you can apply padding or trimming here
                    except Exception as e:
                        print(f"Error loading audio file {audio_path}: {e}")
                        continue

                    # Append to dataset
                    dataset.append({
                        'person_id': person_id,
                        'gender': gender,
                        'sentence_id': sentence_id,
                        'number_in_book': number,
                        'book_id': book_id,
                        'file': file,
                        'audio_path': audio_path,
                        'audio_array': audio_array,  # Keep as NumPy array
                        'sampling_rate': sr,
                        'text': text,
                        'normalized_text': normalized
                    })

        return dataset


In [34]:
from datasets import Features, Value, Array3D, Sequence

# Define the features
features = Features({
    'person_id': Value('string'),
    'gender': Value('string'),
    'sentence_id': Value('int32'),
    'number_in_book': Value('string'),
    'book_id': Value('string'),
    'file': Value('string'),
    'audio_path': Value('string'),
    'audio_array': Sequence(Value('float32')),  # Keep as NumPy array
    'sampling_rate': Value('int32'),
    'text': Value('string'),
    'normalized_text': Value('string'),
})

In [35]:
def convert_to_huggingface_dataset(dataset_list, features):
    # Convert the list of dicts into a Hugging Face Dataset
    dataset = Dataset.from_list(dataset_list, features=features)
    return dataset


In [36]:
if __name__ == "__main__":
    # Step 1: Create the dataset list. Select sample rate here/
    dataset_list = create_tts_dataset(MAIN_DIR, sampling_rate=22050)

    # Step 2: Define features (already done above)

    # Step 3: Convert to Hugging Face Dataset
    tts_dataset = convert_to_huggingface_dataset(dataset_list, features)

    # (Optional) Shuffle the dataset
    tts_dataset = tts_dataset.shuffle(seed=42)

    # Step 4: (Optional) Split into train, validation, test
    # For example, 80% train, 10% validation, 10% test
    train_testvalid = tts_dataset.train_test_split(test_size=0.2, seed=42)
    test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

    dataset_dict = DatasetDict({
        'train': train_testvalid['train'],
        'validation': test_valid['test'],
        'test': test_valid['train']
    })

    # Step 5: Save the dataset to disk (optional)
    dataset_dict.save_to_disk(os.path.join(MAIN_DIR, 'tts_dataset_hf'))
    print("Dataset successfully saved to disk.")

    # Step 6: Loading the dataset later
    # loaded_dataset = DatasetDict.load_from_disk(os.path.join(MAIN_DIR, 'tts_dataset_hf'))

    # Example: Accessing the first instance
    dataset_dict['train'][0]


m1-bot  : folder_name
m3-bot  : folder_name
m2-bot  : folder_name
m4-bot  : folder_name
f1-bot  : folder_name
m5-bot  : folder_name
f6-bot  : folder_name
f5-bot  : folder_name
f4-bot  : folder_name
f3-bot  : folder_name
f2-bot  : folder_name


Saving the dataset (0/1 shards):   0%|          | 0/80 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10 [00:00<?, ? examples/s]

Dataset successfully saved to disk.


In [38]:
dataset_dict['train'][5]

{'person_id': 'f5',
 'gender': 'female',
 'sentence_id': 11,
 'number_in_book': '173',
 'book_id': 'book1',
 'file': 'f5_ID11_173_book1.wav',
 'audio_path': '/content/drive/MyDrive/shayan/turkey_sample/f5-bot/f5_ID11_173_book1.wav',
 'audio_array': [-5.093170329928398e-11,
  -4.729372449219227e-11,
  4.001776687800884e-11,
  3.2741809263825417e-11,
  0.0,
  -2.1827872842550278e-11,
  -6.184563972055912e-11,
  -3.637978807091713e-12,
  9.913492249324918e-11,
  4.001776687800884e-11,
  5.093170329928398e-11,
  5.275069270282984e-11,
  0.0,
  -2.1827872842550278e-11,
  -2.546585164964199e-11,
  5.093170329928398e-11,
  -2.7284841053187847e-11,
  0.0,
  -3.8198777474462986e-11,
  -2.546585164964199e-11,
  -3.637978807091713e-11,
  -2.1827872842550278e-11,
  2.1827872842550278e-11,
  9.822542779147625e-11,
  -3.001332515850663e-11,
  1.6370904631912708e-11,
  2.000888343900442e-11,
  5.4569682106375694e-11,
  5.820766091346741e-11,
  3.637978807091713e-12,
  -4.3655745685100555e-11,
  4.547